# Generación de Embeddings de Texto para LightGBM

**Objetivo**: Extraer embeddings del campo `Description` usando DistilBERT pre-entrenado y agregarlos como features al dataset tabular para LightGBM.

**¿Por qué embeddings en lugar de fine-tuning + blending?**
- LightGBM puede aprender **interacciones** entre features textuales y tabulares
- Un solo modelo, sin problemas de escalas incompatibles entre scores
- Los embeddings pre-entrenados ya capturan semántica rica sin riesgo de overfitting
- PCA reduce la dimensionalidad de 768 a ~32 features manejables

**Pipeline**:
1. Cargar datos con el mismo split que el estudio LGB de Optuna
2. Tokenizar las descripciones con DistilBERT
3. Extraer embeddings del token [CLS] (768 dims)
4. Reducir dimensionalidad con PCA
5. Guardar para integrar con el dataset tabular

#### **Instalar Modulos**

conda install datasets=="2.20.0"

conda install transformers=="4.40.1"

conda install numpy=="1.26.4"

pip install scikit-learn joblib optuna

In [ ]:
# Data processing
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Modeling
import torch
from transformers import DistilBertTokenizerFast, DistilBertModel

# Progress bar
from tqdm.auto import tqdm

# Persistence
from joblib import load, dump


## 1. Cargar datos con el mismo split que LightGBM

In [ ]:
# Paths
BASE_DIR = '../'
PATH_TO_TRAIN = os.path.join(BASE_DIR, "/content/train.csv")
PATH_TO_WORK = os.path.join(BASE_DIR, "work")
PATH_TO_OPTUNA_ARTIFACTS = os.path.join(BASE_DIR, "work/optuna_artifacts")

# Parametros
SEED = 42
N_PCA_COMPONENTS = 32  # Dimensiones finales del embedding (ajustable)
BATCH_SIZE = 16
MAX_LENGTH = 512

In [ ]:
# Cargar datos
df = pd.read_csv(PATH_TO_TRAIN)
print(f'Total de registros: {len(df)}')
print(f'Registros sin descripción: {df["Description"].isnull().sum()}')

# Usar el mismo split que el estudio LGB para comparabilidad
try:
    study_lgb = optuna.create_study(
        direction='maximize',
        storage="sqlite:///../work/db.sqlite3",
        study_name="04 - LGB Multiclass CV",
        load_if_exists=True
    )
    lgb_test_dataset = load(os.path.join(PATH_TO_OPTUNA_ARTIFACTS, get_artifact_filename(study_lgb, 'test')))
    test_pet_ids = lgb_test_dataset.PetID.tolist()

    train_df = df[~df.PetID.isin(test_pet_ids)].reset_index(drop=True)
    test_df = df[df.PetID.isin(test_pet_ids)].reset_index(drop=True)
    print(f'Usando split de Optuna LGB')
except Exception as e:
    print(f'No se encontró el estudio LGB ({e}), usando train_test_split')
    train_df, test_df = train_test_split(df, test_size=0.2, random_state=SEED, stratify=df.AdoptionSpeed)

print(f'Train: {len(train_df)}, Test: {len(test_df)}')

Total de registros: 14993
Registros sin descripción: 13
No se encontró el estudio LGB (name 'optuna' is not defined), usando train_test_split
Train: 11994, Test: 2999


# Pasos para generar embeddings para modelo tabular

## 2. Cargar modelo DistilBERT pre-entrenado

Usamos el modelo **pre-entrenado** (sin fine-tuning) para extraer embeddings.
Esto evita el overfitting y los embeddings genéricos ya capturan buena semántica.

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
model = DistilBertModel.from_pretrained('distilbert-base-uncased')
model = model.to(device)
model.eval()
print(f'Modelo cargado - dimensión del embedding: {model.config.hidden_size}')

## 3. Función para extraer embeddings

Extraemos el embedding del token **[CLS]** (primera posición), que representa un resumen de toda la secuencia.

In [ ]:
def extract_embeddings(texts, tokenizer, model, device, batch_size=64, max_length=512):
    """Extrae embeddings [CLS] de DistilBERT para una lista de textos.

    Args:
        texts: Lista de strings
        tokenizer: Tokenizer de DistilBERT
        model: Modelo DistilBERT
        device: 'cuda' o 'cpu'
        batch_size: Tamaño del batch
        max_length: Longitud máxima de tokenización

    Returns:
        numpy array de shape (n_texts, 768)
    """
    all_embeddings = []

    for i in tqdm(range(0, len(texts), batch_size), desc='Extrayendo embeddings'):
        batch_texts = texts[i:i+batch_size]

        # Tokenizar
        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors='pt'
        ).to(device)

        # Extraer embeddings sin calcular gradientes
        with torch.no_grad():
            outputs = model(**encoded)

        # Tomar el embedding del token [CLS] (posición 0)
        cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        all_embeddings.append(cls_embeddings)

    return np.vstack(all_embeddings)

## 4. Extraer embeddings para train y test

Para las descripciones nulas, usamos un string vacío (DistilBERT generará un embedding "neutral").

In [ ]:
# Preparar textos (reemplazar NaN con string vacío)
train_texts = train_df['Description'].fillna('').tolist()
test_texts = test_df['Description'].fillna('').tolist()

print(f'Extrayendo embeddings para {len(train_texts)} textos de train...')
train_embeddings = extract_embeddings(train_texts, tokenizer, model, device, BATCH_SIZE, MAX_LENGTH)
print(f'Shape train embeddings: {train_embeddings.shape}')

print(f'\nExtrayendo embeddings para {len(test_texts)} textos de test...')
test_embeddings = extract_embeddings(test_texts, tokenizer, model, device, BATCH_SIZE, MAX_LENGTH)
print(f'Shape test embeddings: {test_embeddings.shape}')

## 5. Reducir dimensionalidad con PCA

768 features es demasiado para LightGBM (riesgo de overfitting y lentitud).
PCA reduce a N_PCA_COMPONENTS manteniendo la mayor varianza posible.

In [ ]:
# Estandarizar antes de PCA
scaler = StandardScaler()
train_embeddings_scaled = scaler.fit_transform(train_embeddings)
test_embeddings_scaled = scaler.transform(test_embeddings)

# Aplicar PCA
pca = PCA(n_components=N_PCA_COMPONENTS, random_state=SEED)
train_pca = pca.fit_transform(train_embeddings_scaled)
test_pca = pca.transform(test_embeddings_scaled)

explained_var = pca.explained_variance_ratio_.sum()
print(f'PCA: {N_PCA_COMPONENTS} componentes explican {explained_var:.1%} de la varianza')
print(f'Shape train PCA: {train_pca.shape}')
print(f'Shape test PCA: {test_pca.shape}')

In [ ]:
# Visualizar varianza explicada acumulada
import matplotlib.pyplot as plt

cumvar = np.cumsum(pca.explained_variance_ratio_)
plt.figure(figsize=(10, 5))
plt.bar(range(1, N_PCA_COMPONENTS+1), pca.explained_variance_ratio_, alpha=0.6, label='Individual')
plt.step(range(1, N_PCA_COMPONENTS+1), cumvar, where='mid', label='Acumulada', color='red')
plt.xlabel('Componente PCA')
plt.ylabel('Varianza explicada')
plt.title(f'PCA sobre embeddings DistilBERT - {explained_var:.1%} varianza total')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Crear DataFrames con embeddings

In [ ]:
# Crear columnas con nombres descriptivos
emb_cols = [f'text_emb_{i}' for i in range(N_PCA_COMPONENTS)]

train_emb_df = pd.DataFrame(train_pca, columns=emb_cols)
train_emb_df['PetID'] = train_df['PetID'].values

test_emb_df = pd.DataFrame(test_pca, columns=emb_cols)
test_emb_df['PetID'] = test_df['PetID'].values

# Combinar train + test embeddings en un solo DataFrame
all_emb_df = pd.concat([train_emb_df, test_emb_df], ignore_index=True)
print(f'DataFrame de embeddings: {all_emb_df.shape}')
all_emb_df.head()

## 7. Guardar artefactos

In [ ]:
# Guardar embeddings, scaler y PCA
os.makedirs(PATH_TO_WORK, exist_ok=True)

emb_path = os.path.join(PATH_TO_WORK, 'text_embeddings_pca.joblib')
dump(all_emb_df, emb_path)
print(f'Embeddings guardados en: {emb_path}')

scaler_path = os.path.join(PATH_TO_WORK, 'text_emb_scaler.joblib')
dump(scaler, scaler_path)
print(f'Scaler guardado en: {scaler_path}')

pca_path = os.path.join(PATH_TO_WORK, 'text_emb_pca.joblib')
dump(pca, pca_path)
print(f'PCA guardado en: {pca_path}')

## 8. Demo: Integrar con dataset tabular y entrenar LightGBM

Ejemplo rápido de cómo usar los embeddings con LightGBM.

In [ ]:
from sklearn.metrics import cohen_kappa_score

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    print('LightGBM no instalado. Instalar con: pip install lightgbm')
    HAS_LGB = False

In [ ]:
if HAS_LGB:
    # Features tabulares originales
    feature_cols = ['Type', 'Age', 'Breed1', 'Breed2', 'Gender', 'Color1', 'Color2', 'Color3',
                    'MaturitySize', 'FurLength', 'Vaccinated', 'Dewormed', 'Sterilized',
                    'Health', 'Quantity', 'Fee', 'State', 'VideoAmt', 'PhotoAmt']
    target_col = 'AdoptionSpeed'

    # --- Modelo SIN embeddings ---
    X_train = train_df[feature_cols].copy()
    X_test = test_df[feature_cols].copy()
    y_train = train_df[target_col]
    y_test = test_df[target_col]

    # Rellenar PhotoAmt NaN
    X_train['PhotoAmt'] = X_train['PhotoAmt'].fillna(0)
    X_test['PhotoAmt'] = X_test['PhotoAmt'].fillna(0)

    model_base = lgb.LGBMClassifier(
        n_estimators=500, learning_rate=0.05, num_leaves=31,
        random_state=SEED, verbose=-1, n_jobs=-1
    )
    model_base.fit(X_train, y_train)
    pred_base = model_base.predict(X_test)
    kappa_base = cohen_kappa_score(y_test, pred_base, weights='quadratic')
    print(f'LightGBM SIN embeddings - QWK: {kappa_base:.4f}')

    # --- Modelo CON embeddings ---
    X_train_emb = pd.concat([X_train.reset_index(drop=True), train_emb_df[emb_cols].reset_index(drop=True)], axis=1)
    X_test_emb = pd.concat([X_test.reset_index(drop=True), test_emb_df[emb_cols].reset_index(drop=True)], axis=1)

    model_emb = lgb.LGBMClassifier(
        n_estimators=500, learning_rate=0.05, num_leaves=31,
        random_state=SEED, verbose=-1, n_jobs=-1
    )
    model_emb.fit(X_train_emb, y_train)
    pred_emb = model_emb.predict(X_test_emb)
    kappa_emb = cohen_kappa_score(y_test, pred_emb, weights='quadratic')
    print(f'LightGBM CON embeddings - QWK: {kappa_emb:.4f}')

    print(f'\nMejora: {kappa_emb - kappa_base:+.4f} ({(kappa_emb/kappa_base - 1)*100:+.1f}%)')

In [ ]:
if HAS_LGB:
    # Feature importance de las top 30 features
    importances = pd.DataFrame({
        'feature': X_train_emb.columns,
        'importance': model_emb.feature_importances_
    }).sort_values('importance', ascending=True).tail(30)

    plt.figure(figsize=(10, 8))
    colors = ['#2196F3' if 'text_emb' in f else '#FF9800' for f in importances['feature']]
    plt.barh(importances['feature'], importances['importance'], color=colors)
    plt.xlabel('Importance')
    plt.title('Top 30 Feature Importances (Naranja=Tabular, Azul=Embedding)')
    plt.tight_layout()
    plt.show()

## 9. Cómo integrar en tu notebook de LightGBM

Para usar estos embeddings en tu pipeline existente:

```python
from joblib import load

# Cargar embeddings pre-calculados
emb_df = load('../work/text_embeddings_pca.joblib')

# Merge con tu dataset tabular por PetID
train_df = train_df.merge(emb_df, on='PetID', how='left')
test_df = test_df.merge(emb_df, on='PetID', how='left')

# Agregar las columnas text_emb_0 ... text_emb_31 a tus features
emb_cols = [c for c in emb_df.columns if c.startswith('text_emb_')]
feature_cols = feature_cols + emb_cols

# Rellenar NaN para mascotas sin embedding
train_df[emb_cols] = train_df[emb_cols].fillna(0)
test_df[emb_cols] = test_df[emb_cols].fillna(0)
```


# 10. Clasificación con DistilBERT (Fine-tuning) — Evaluación QWK

Siguiendo el enfoque del notebook del profesor `06_text_classification`,
hacemos **fine-tuning** de `AutoModelForSequenceClassification` (DistilBERT)
directamente sobre las descripciones de texto.

Evaluamos con **Quadratic Weighted Kappa (QWK)** como métrica principal.

**Nota**: Esta sección usa directamente el texto crudo (Description),
NO los embeddings PCA generados anteriormente. Los embeddings PCA
quedan guardados en `work/` para uso con LightGBM u otros modelos tabulares.

In [ ]:
from datasets import Dataset, DatasetDict
from torch.utils.data import DataLoader
from transformers import (
    DistilBertTokenizerFast,
    DataCollatorWithPadding,
    AutoModelForSequenceClassification,
    get_scheduler
)
from torch.optim import AdamW
from sklearn.metrics import cohen_kappa_score, classification_report

### 10.1 Preparar datasets para fine-tuning

Convertimos los DataFrames de train/test (ya cargados en la sección 1)
a formato HuggingFace `Dataset`, filtrando registros sin descripción.

In [ ]:

train_text_df = train_df[train_df['Description'].notnull()].copy()
test_text_df = test_df[test_df['Description'].notnull()].copy()

# Crear columna 'labels' (requerida por HuggingFace)
train_text_df['labels'] = train_text_df['AdoptionSpeed']
test_text_df['labels'] = test_text_df['AdoptionSpeed']

print(f'Train (con descripcion): {len(train_text_df)}')
print(f'Test (con descripcion): {len(test_text_df)}')

# Convertir a Dataset de HuggingFace
train_dataset = Dataset.from_pandas(train_text_df)
test_dataset = Dataset.from_pandas(test_text_df)

dataset = DatasetDict({
    'train': train_dataset,
    'test': test_dataset
})

# Codificar la columna de etiquetas como clases
dataset = dataset.class_encode_column('labels')

# Columnas a remover antes de la tokenizacion
cols_to_remove = [col for col in dataset['train'].column_names if col != 'labels']
print(f'\nColumnas a remover: {cols_to_remove}')

Train (con descripcion): 11984
Test (con descripcion): 2996


Stringifying the column:   0%|          | 0/11984 [00:00<?, ? examples/s]

Casting to class labels:   0%|          | 0/11984 [00:00<?, ? examples/s]

Stringifying the column:   0%|          | 0/2996 [00:00<?, ? examples/s]

Casting to class labels:   0%|          | 0/2996 [00:00<?, ? examples/s]


Columnas a remover: ['Type', 'Name', 'Age', 'Breed1', 'Breed2', 'Gender', 'Color1', 'Color2', 'Color3', 'MaturitySize', 'FurLength', 'Vaccinated', 'Dewormed', 'Sterilized', 'Health', 'Quantity', 'Fee', 'State', 'RescuerID', 'VideoAmt', 'Description', 'PetID', 'PhotoAmt', 'AdoptionSpeed', '__index_level_0__']


In [ ]:
# Verificar las clases
class_label = dataset['train'].features['labels']
classes = class_label.names
num_labels = class_label.num_classes
print(f'Clases: {classes}')
print(f'Numero de clases: {num_labels}')

Clases: ['0', '1', '2', '3', '4']
Numero de clases: 5


### 10.2 Tokenizar descripciones

In [ ]:
# Reiniciar el tokenizer (ya cargado arriba, pero por claridad)
tokenizer_ft = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

# Funcion de tokenizacion
def tokenize(batch):
    return tokenizer_ft(
        batch['Description'],
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH
    )

# Tokenizar y remover columnas originales
dataset_enc = dataset.map(tokenize, batched=True, remove_columns=cols_to_remove)

# Formato PyTorch
dataset_enc.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

print(f'Columnas del dataset tokenizado: {dataset_enc["train"].column_names}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/11984 [00:00<?, ? examples/s]

Map:   0%|          | 0/2996 [00:00<?, ? examples/s]

Columnas del dataset tokenizado: ['labels', 'input_ids', 'attention_mask']


### 10.3 Crear DataLoaders

In [ ]:
# Data collator con padding dinamico
data_collator = DataCollatorWithPadding(tokenizer=tokenizer_ft)

# DataLoaders
train_dataloader = DataLoader(
    dataset_enc['train'],
    shuffle=True,
    batch_size=BATCH_SIZE,
    collate_fn=data_collator
)
eval_dataloader = DataLoader(
    dataset_enc['test'],
    batch_size=BATCH_SIZE,
    collate_fn=data_collator
)

print(f'Batches de entrenamiento: {len(train_dataloader)}')
print(f'Batches de evaluacion: {len(eval_dataloader)}')

Batches de entrenamiento: 749
Batches de evaluacion: 188


### 10.4 Cargar modelo y configurar entrenamiento

In [ ]:
import torch
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

# Cargar modelo de clasificacion
model_ft = AutoModelForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=num_labels
)
model_ft.to(device)
print(f'Modelo cargado en: {device}')
print(model_ft)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Modelo cargado en: cpu
DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout

In [ ]:
# Parametros de entrenamiento
learning_rate = 0.001
num_epochs = 15

# Optimizador
optimizer = AdamW(model_ft.parameters(), lr=learning_rate)

# Learning rate scheduler (linear decay)
num_training_batches = len(train_dataloader)
num_training_steps = num_epochs * num_training_batches
lr_scheduler = get_scheduler(
    'linear',
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

print(f'Epocas: {num_epochs}')
print(f'Batches por epoca: {num_training_batches}')
print(f'Steps totales: {num_training_steps}')

Epocas: 15
Batches por epoca: 749
Steps totales: 11235


### 10.5 Entrenar el modelo

In [ ]:
# Training loop
progress_bar = tqdm(range(num_training_steps))

model_ft.train()
for epoch in range(num_epochs):
    for batch in train_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model_ft(**batch)
        loss = outputs.loss
        loss.backward()

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)

  0%|          | 0/11235 [00:00<?, ?it/s]

### 10.7 Guardar el modelo entrenado

In [ ]:
# Crear un directorio para guardar el modelo fine-tuned
model_save_path = os.path.join(PATH_TO_WORK, 'distilbert_finetuned')
os.makedirs(model_save_path, exist_ok=True)

# Guardar el modelo y el tokenizer
model_ft.save_pretrained(model_save_path)
tokenizer_ft.save_pretrained(model_save_path)

print(f'Modelo y tokenizer guardados en: {model_save_path}')

### 10.6 Evaluar con Quadratic Weighted Kappa (QWK)

In [ ]:
# Evaluacion
all_predictions = []
all_labels = []

model_ft.eval()
for batch in eval_dataloader:
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = model_ft(**batch)

    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)

    all_predictions.extend(predictions.cpu().numpy())
    all_labels.extend(batch['labels'].cpu().numpy())

all_predictions = np.array(all_predictions)
all_labels = np.array(all_labels)

# Quadratic Weighted Kappa
qwk = cohen_kappa_score(all_labels, all_predictions, weights='quadratic')

print(f'Quadratic Weighted Kappa: {qwk:.4f}')
print(f'\nClassification Report:')
print(classification_report(
    all_labels, all_predictions,
    target_names=[f'Clase {i}' for i in range(num_labels)]
))

### 10.8 Predicción de ejemplo

In [ ]:
# Prediccion de un ejemplo (igual que el profesor)
example_idx = 4
desc = test_text_df.iloc[example_idx]['Description']
real_label = test_text_df.iloc[example_idx]['AdoptionSpeed']

print(f'Descripcion: {desc}')
print(f'AdoptionSpeed real: {real_label}')

# Tokenizar
inputs = tokenizer_ft(
    desc, padding=True, truncation=True, return_tensors='pt'
).to(device)

# Inferencia
model_ft.eval()
with torch.no_grad():
    outputs = model_ft(**inputs)

# Probabilidades
probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)
probs = probabilities.detach().cpu().numpy()[0]
predicted_class = np.argmax(probs)

print(f'\nAdoptionSpeed predicho: {predicted_class}')
print(f'\nProbabilidades por clase:')
np.set_printoptions(suppress=True, formatter={'float_kind': '{:.8f}'.format})
for i, prob in enumerate(probs):
    marker = ' <-- prediccion' if i == predicted_class else ''
    print(f'  Clase {i}: {prob:.4f}{marker}')